# 08 · Mapa geográfico  *(sección 8 del script)*

**Qué hacemos:** miramos el análisis **en el mapa**, con tres piezas:

1. **Mapa de puntos interactivo (Plotly)** sobre una muestra de 5.000 clientes, coloreado por `review_score`.
2. **Mapa de calor de densidad (Folium + OpenStreetMap)** con el dataset completo.
3. **Tabla resumen por estado** (los 10 con más pedidos): pedidos, review promedio y % de demora.

**Para qué:** para ver si hay un **patrón geográfico en la satisfacción y en el volumen de pedidos**, más allá de lo que muestran los promedios agregados por estado — y para respaldar en números lo que se ve en los mapas (los datos de la tabla son los que entran a la hipótesis 3 del notebook 10).

## Celda estándar: carga del artefacto + guardado de figuras

In [1]:
# Celda estándar (detallada en 01_configuracion_inicial.ipynb)
%matplotlib inline

import re
import unicodedata
import warnings
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore")

BASE = Path.cwd()
if not (BASE / "data").exists() and (BASE.parent / "data").exists():
    BASE = BASE.parent

CSV_UNIFICADO = BASE / "data" / "olist_dataset_unificado.csv"
PIPELINE_DIR = BASE / "data" / "pipeline"
FIGS_DIR = BASE / "figuras_eda"

OLIST_BLUE = "#0A4EE4"
OLIST_BLUE_DARK = "#0D366B"
OLIST_GRAY = "#52514E"
PALETA_CATEGORICA = ["#0A4EE4", "#eb6834", "#1baf7a", "#eda100",
                     "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
SUDESTE = {"SP", "RJ", "MG", "ES"}

sns.set_theme(style="whitegrid", rc={
    "axes.edgecolor": "#c3c2b7",
    "axes.labelcolor": OLIST_GRAY,
    "text.color": "#0b0b0b",
    "xtick.color": OLIST_GRAY,
    "ytick.color": OLIST_GRAY,
    "font.family": "sans-serif",
})
plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.titleweight"] = "bold"
pd.set_option("display.max_columns", 50)

# --- Carga del artefacto del notebook 04 ---
COLUMNAS_FECHA = [
    "shipping_limit_date", "order_purchase_timestamp", "order_approved_at",
    "order_delivered_carrier_date", "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

RUTA_FEATURES = PIPELINE_DIR / "02_df_features.csv"
if not RUTA_FEATURES.exists():
    raise FileNotFoundError(
        f"No existe {RUTA_FEATURES}. Ejecutá primero los notebooks 02, 03 y 04."
    )

df = pd.read_csv(RUTA_FEATURES)
for col in COLUMNAS_FECHA:
    df[col] = pd.to_datetime(df[col], errors="coerce")
df["mes_compra"] = df["order_purchase_timestamp"].dt.to_period("M").dt.to_timestamp()

# --- Guardado de figuras de este notebook ---
def configurar_guardado(prefijo):
    FIGS_DIR.mkdir(exist_ok=True)
    contador = {"n": 0}

    def guardar():
        contador["n"] += 1
        ruta = FIGS_DIR / f"{prefijo}_{contador['n']:02d}.png"
        plt.savefig(ruta, dpi=110, bbox_inches="tight")
        return ruta

    return guardar


guardar_fig = configurar_guardado("nb08")
print(f"filas={df.shape[0]:,} columnas={df.shape[1]} | figuras -> {FIGS_DIR}")

filas=112,650 columnas=43 | figuras -> D:\Ciencia de Datos\Proyecto Integrador\trabajo_ integrador_versionNati\trabajo_ integrador_versionNati\figuras_eda


## Mapa 1 — Puntos de clientes por satisfacción (Plotly)

**Qué hacemos:** mapa interactivo de dispersión con `customer_lat`/`customer_lng` sobre una **muestra de 5.000 clientes** (filas con coordenadas y puntaje válidos), coloreado por `review_score` (escala de azules) y con *hover* de estado, ciudad, si se demoró y la distancia.

**Para qué:** para ver si hay un **patrón geográfico en la satisfacción** — zonas sistemáticamente más azul claro (menor puntaje) — más allá de los promedios por estado.

> El `try/except AttributeError` es un adaptador de versión: Plotly renombró `scatter_mapbox` → `scatter_map`; el código usa la función nueva y, si no existe, cae a la vieja.

In [2]:
import plotly.express as px

validos = df.dropna(subset=["customer_lat", "customer_lng", "review_score"])
muestra = validos.sample(n=min(5000, len(validos)), random_state=42)

try:
    fig = px.scatter_map(
        muestra, lat="customer_lat", lon="customer_lng",
        color="review_score", color_continuous_scale="Blues", zoom=3,
        hover_data=["customer_state", "customer_city", "envio_demorado", "distancia_km"],
        title="Ubicación de clientes (muestra) por review_score",
    )
except AttributeError:
    fig = px.scatter_mapbox(
        muestra, lat="customer_lat", lon="customer_lng",
        color="review_score", color_continuous_scale="Blues", zoom=3,
        mapbox_style="open-street-map",
        hover_data=["customer_state", "customer_city", "envio_demorado", "distancia_km"],
        title="Ubicación de clientes (muestra) por review_score",
    )
fig.show()

**Insight:** los clientes se **concentran fuertemente en el eje Sudeste** (São Paulo, Río de Janeiro, Minas Gerais) — la misma zona donde se concentran los vendedores (ya intuido en el notebook 07). Esa superposición es justo lo que hace que la distancia intra-Sudeste sea corta y la del resto del país, larga.

## Mapa 2 — Densidad de pedidos (Folium + HeatMap)

**Qué hacemos:** mapa de calor de densidad de pedidos sobre **el dataset completo** (no una muestra), centrado en Brasil (`[-15, -50]`, zoom 4), con `HeatMap` de la librería `folium`.

**Para qué:** complementar el mapa anterior con una **vista de volumen** (dónde hay más pedidos, no solo dónde están coloreados por puntaje), sin el límite de los 5.000 puntos.

In [3]:
import folium
from folium.plugins import HeatMap

puntos = df.dropna(subset=["customer_lat", "customer_lng"])[["customer_lat", "customer_lng"]]
mapa_calor = folium.Map(location=[-15.0, -50.0], zoom_start=4, tiles="OpenStreetMap")
HeatMap(puntos.values.tolist(), radius=10, blur=8).add_to(mapa_calor)
mapa_calor

**Insight:** el mapa de calor **confirma la misma concentración geográfica** que el mapa de puntos, ahora **sin el sesgo de tomar solo una muestra**: el litoral Sudeste "enciende" el mapa, mientras que el interior y el norte quedan muy tenues — poca densidad de pedidos y, de paso, las rutas más largas hacia esos destinos.

## Tabla resumen por estado

**Qué hacemos:** tabla de los **10 estados con más pedidos**, con cantidad de pedidos, `review_score` promedio y % de envíos demorados.

**Para qué:** **respaldar en números** lo que se ve en los mapas, y tener los datos concretos que se usan en la hipótesis 3 (notebook 10) sobre la brecha geográfica.

In [4]:
resumen_estado = (
    df.groupby("customer_state")
    .agg(pedidos=("order_id", "nunique"),
         review_score_prom=("review_score", "mean"),
         pct_demorado=("envio_demorado", "mean"))
    .sort_values("pedidos", ascending=False)
    .head(10)
    .round(2)
)
resumen_estado["pct_demorado"] = (resumen_estado["pct_demorado"] * 100).round(1)
resumen_estado

,pedidos,review_score_prom,pct_demorado
customer_state,,,
SP,41375,4.13,6.0
RJ,12762,3.81,13.0
MG,11544,4.09,5.0
RS,5432,4.05,7.0
PR,4998,4.11,5.0
SC,3612,4.00,10.0
BA,3358,3.82,14.0
DF,2125,4.00,7.0
ES,2025,3.99,12.0


**Insight:** los estados **fuera del eje Sudeste**, aunque tienen menos pedidos, muestran un `review_score` promedio **algo más bajo** y un % de envíos demorados **más alto** que SP/RJ/MG — la distancia a los vendedores parece pesar en la experiencia de entrega (se contrasta con test estadístico en el notebook 10).

**Siguiente paso:** `09_nlp_sobre_las_resenas.ipynb` — leer lo que los clientes escriben en sus reseñas (en portugués).